# 01 · Muestreo de PopResume y OCR con PaddleOCR (Colab, GPU T4)

**Etapa 1 del pipeline.** Este notebook:

1. baja los metadatos de PopResume y saca una muestra estratificada de **800 CVs por ocupación (4 000)** con semilla fija;
2. descarga **solo** las imágenes sin foto (`has_photo == False`) de esos 4 000 CVs;
3. corre PaddleOCR 3.x sobre cada imagen y guarda **un JSON por CV** con texto, confianza y caja por línea, ya en orden de lectura;
4. reporta tiempo por CV, hardware, confianza media e imágenes fallidas.

### Antes de ejecutar

- **Entorno de ejecución → Cambiar tipo de entorno → GPU T4.**
- Ícono 🔑 (Secretos) a la izquierda: crea `HF_TOKEN` con tu token *Read* de Hugging Face y activa «Acceso del notebook».
- Haber aceptado las condiciones del dataset en https://huggingface.co/datasets/sumin-yu/PopResume.
- Cambia `TU_USUARIO` en la celda de entorno por tu usuario de GitHub.

Todo se guarda en `MyDrive/cv_experience_ocr_bert/`. Si Colab corta la sesión, vuelve a ejecutar
todo: la descarga y el OCR **se reanudan** donde quedaron.

> Este notebook **no importa PyTorch**: PaddlePaddle-GPU y el PyTorch de Colab traen librerías CUDA
> distintas. Los notebooks 03–05 se ejecutan en un entorno nuevo
> (*Entorno de ejecución → Desconectar y borrar entorno*).

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

## Instalación (versiones fijadas)

Misma receta que la clase de la sesión 10 (`00_colab_clase.ipynb`): `paddlepaddle-gpu` desde el índice
oficial de CUDA 12.6 y **nunca** junto con `paddlepaddle` (CPU), porque conviven sin error y todo corre
en CPU sin avisar. Tarda ~3 minutos.

In [ ]:
!pip install -q paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ --extra-index-url https://pypi.org/simple
!pip install -q "paddleocr==3.7.0" "datasets>=3" "huggingface_hub>=0.30" "pyarrow>=15"
print("Instalación terminada.")

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/TU_USUARIO/cv_experience_ocr_bert.git"   # ← cambia TU_USUARIO
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/cv_experience_ocr_bert")
    if not (RAIZ / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(RAIZ)], check=True)
    else:
        subprocess.run(["git", "-C", str(RAIZ), "pull", "--ff-only"], check=False)
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    HF_TOKEN = os.environ.get("HF_TOKEN")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from huggingface_hub import login
login(token=HF_TOKEN)
print("Raíz del proyecto:", RAIZ)

## El doctor: ¿PaddlePaddle ve la GPU?

PaddlePaddle **no falla** si la GPU no está disponible: cae a CPU en silencio y todo corre ~40 veces
más lento. Comprobamos antes de gastar una hora.

In [ ]:
import paddle
print("Paddle", paddle.__version__, "· compilado con CUDA:", paddle.device.is_compiled_with_cuda())
print("GPUs visibles:", paddle.device.cuda.device_count())
DEVICE = "gpu:0" if paddle.device.cuda.device_count() > 0 else "cpu"
GPU_NOMBRE = paddle.device.cuda.get_device_name(0) if DEVICE != "cpu" else "CPU"
print("Dispositivo:", DEVICE, "·", GPU_NOMBRE)
assert DEVICE != "cpu", "Sin GPU: activa la T4 en «Cambiar tipo de entorno de ejecución»."

## 1 · Muestreo estratificado

- Se baja la config `text` (una fila por CV, liviana).
- 800 CVs por ocupación; dentro de cada una, asignación **proporcional por banda de experiencia**
  (0–2, 3–5, 6–10, 11–20) para que la muestra conserve la distribución del target.
- Semilla fija (`SEED = 42`); la lista de IDs se guarda en `data/sample_ids.csv`.

In [ ]:
import json
import numpy as np
import pandas as pd
from src import config, data

meta = data.cargar_metadatos()
print(f"Metadatos: {len(meta):,} CVs")
print(meta["job"].value_counts())

muestra = data.muestrear(meta)
muestra[["id", "job"]].to_csv(config.DATA / "sample_ids.csv", index=False)
muestra.to_parquet(config.PROCESADOS / "muestra_meta.parquet", index=False)
print("\nMuestra por ocupación:")
print(muestra["job"].value_counts())

### Target y tope

`exp_year` se recorta a 0–20. Si más del 15 % de la muestra queda en el tope, se sube a 25 o 30
(y el mismo tope vale para todos los modelos).

In [ ]:
y = muestra[config.TARGET].astype(float)
TOPE = data.elegir_tope(y)
en_tope = (y >= TOPE).mean()
print(f"exp_year: min {y.min():.0f} · mediana {y.median():.0f} · máx {y.max():.0f}")
print(f"CVs con exp_year ≥ 20: {(y >= 20).mean():.1%} → TOPE = {TOPE} (en el tope: {en_tope:.1%})")
run = {"seed": config.SEED, "n_por_ocupacion": config.N_POR_OCUPACION, "n_total": len(muestra),
       "tope": TOPE, "pct_en_tope_20": float((y >= 20).mean()), "pct_en_tope_final": float(en_tope)}
(config.PROCESADOS / "run_config.json").write_text(json.dumps(run, indent=2))

dist = (muestra.assign(y=y.clip(0, TOPE), banda=y.clip(0, TOPE).map(config.banda).map(dict(enumerate(config.NOMBRES_BANDAS))))
        .pivot_table(index="job", columns="banda", values="id", aggfunc="count", fill_value=0))
dist.to_csv(config.TABLAS / "muestra_por_banda.csv")
dist

## 2 · Descarga de las imágenes sin foto

Los parquet de imágenes pesan ~16–20 GB en total. En lugar de bajarlo todo, se lee por *row group*:
primero las columnas `id` y `has_photo` (pocos KB) y solo se baja `image` donde hay CVs de la muestra.
Las 4 000 imágenes se guardan como PNG en `data/raw/images/` (en Drive, **no** en el repo).

In [ ]:
IMAGENES = config.DATA / "raw" / "images"
data.descargar_imagenes(muestra["id"], IMAGENES, token=HF_TOKEN)
print("PNG en disco:", len(list(IMAGENES.glob("*.png"))))

## 3 · El modelo de OCR

**Elección:** `PP-OCRv5_mobile_det` + `en_PP-OCRv5_mobile_rec`.

- Los CVs de PopResume son renders digitales limpios y en inglés: no hay escritura a mano ni fotos
  torcidas, que es donde el modelo *server* marca diferencia (sesión 10, §5–6).
- El reconocedor `en_` está entrenado solo para inglés: menos confusiones con alfabetos ajenos (en la clase
  el multilingüe leyó un adorno como «の»).
- La app vuelve a correr **el mismo** modelo en una laptop sin GPU para el OCR en vivo; *mobile* lo hace viable.
  Usar otro modelo en la app introduciría un cambio de distribución entre entrenamiento y uso.

Se apagan orientación de página, *unwarping* y orientación de línea: los renders ya vienen derechos.

In [ ]:
import time
from src import ocr as ocrmod

rutas = sorted(IMAGENES.glob("*.png"))
ocr = ocrmod.crear_ocr(device=DEVICE)
ocr.predict(str(rutas[0]))                       # calentamiento: no se cronometra
t0 = time.perf_counter()
lineas, alto, ancho = ocrmod.leer_imagen(ocr, rutas[1])
print(f"Una página: {time.perf_counter() - t0:.2f} s · {len(lineas)} líneas · {ancho}×{alto}px")
ordenadas, dos_col = ocrmod.ordenar_lectura(lineas, ancho)
print("¿Dos columnas?", dos_col)
print(ocrmod.texto_de(ordenadas)[:1500])

### Orden de lectura

PaddleOCR devuelve las líneas aproximadamente por altura. En un CV a **dos columnas** eso intercala la
barra lateral con la experiencia (el mismo problema del *Mercurio Peruano* en la clase). `ordenar_lectura`:

1. busca un **canal vertical** entre el 25 % y el 75 % del ancho que casi ninguna caja cruce (≤ 5 %) y con
   al menos 20 % de las líneas a cada lado → página a dos columnas;
2. las líneas que **cruzan** el canal (nombre, títulos a todo el ancho) parten la página en franjas;
3. en cada franja se lee la columna izquierda completa y luego la derecha;
4. dentro de cada columna, las cajas cuyos centros están a menos de media altura forman una fila,
   que se lee de izquierda a derecha.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

def dibujar(ruta, lineas, eje, titulo=""):
    eje.imshow(Image.open(ruta).convert("RGB"))
    for k, l in enumerate(lineas):
        x0, y0, x1, y1 = l["box"]
        color = "#0ca30c" if l["conf"] >= 0.9 else "#fab219" if l["conf"] >= 0.7 else "#d03b3b"
        eje.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=1, edgecolor=color))
        eje.text(x0, y0, str(k), fontsize=5, color="#2a78d6")
    eje.set_title(titulo); eje.axis("off")

fig, eje = plt.subplots(figsize=(7, 9))
dibujar(rutas[1], ordenadas, eje, "Cajas de PaddleOCR (número = orden de lectura; color = confianza)")
fig.savefig(config.FIGURAS / "ocr_ejemplo_cajas.png", dpi=130, bbox_inches="tight")
plt.show()
(config.TABLAS / "ocr_ejemplo.json").write_text(json.dumps(
    {"id": rutas[1].stem, "dos_columnas": dos_col, "texto": ocrmod.texto_de(ordenadas)}, ensure_ascii=False))

## 4 · OCR de los 4 000 CVs

Un JSON por CV en `data/raw/ocr_json/` con `lines: [{text, conf, box}]` en orden de lectura,
`two_columns`, `seconds` y `error`. Reanudable.

In [ ]:
JSONS = config.DATA / "raw" / "ocr_json"
t0 = time.perf_counter()
hechos, fallidos = ocrmod.procesar_lote(ocr, rutas, JSONS)
print(f"Nuevos: {hechos} · fallidos: {fallidos} · {time.perf_counter() - t0:.0f} s en esta sesión")

## 5 · Reporte del OCR

In [ ]:
registros = ocrmod.cargar_json(JSONS)
ok = [r for r in registros if not r["error"]]
filas = []
for r in ok:
    conf = [l["conf"] for l in r["lines"]]
    filas.append({"id": r["id"], "n_lineas": len(conf), "conf_media": float(np.mean(conf)) if conf else np.nan,
                  "pct_lineas_conf<0.9": float(np.mean(np.array(conf) < 0.9)) if conf else np.nan,
                  "dos_columnas": r["two_columns"], "segundos": r["seconds"],
                  "texto_ocr": ocrmod.texto_de(r["lines"]),
                  "lineas_json": json.dumps(r["lines"], ensure_ascii=False)})
ocr_df = pd.DataFrame(filas).merge(muestra[["id", "job"]], on="id")
ocr_df.to_parquet(config.PROCESADOS / "ocr.parquet", index=False)

resumen_ocr = {
    "hardware": GPU_NOMBRE, "paddleocr": __import__("paddleocr").__version__, "paddlepaddle": paddle.__version__,
    "modelo_det": config.OCR_DET, "modelo_rec": config.OCR_REC,
    "cvs_procesados": len(ok), "imagenes_fallidas": len(registros) - len(ok),
    "segundos_por_cv_media": float(ocr_df["segundos"].mean()),
    "segundos_por_cv_mediana": float(ocr_df["segundos"].median()),
    "confianza_media": float(ocr_df["conf_media"].mean()),
    "pct_cvs_dos_columnas": float(ocr_df["dos_columnas"].mean()),
    "lineas_por_cv_media": float(ocr_df["n_lineas"].mean()),
}
(config.TABLAS / "ocr_resumen.json").write_text(json.dumps(resumen_ocr, indent=2))
display(pd.Series(resumen_ocr).to_frame("valor"))
display(ocr_df.groupby("job")[["conf_media", "segundos", "dos_columnas", "n_lineas"]].mean().round(3))

### 50 textos OCR para leer a mano

El issue pide **leer al menos 50** textos OCR antes de definir el enmascaramiento. Se escriben a
`output/ocr_muestra_50.txt` (10 por ocupación, semilla fija). Léelos buscando frases tipo
«X years of experience», «Age: 34» o fechas de nacimiento: el notebook 02 las enmascara.

In [ ]:
lectura = ocr_df.groupby("job", group_keys=False).apply(lambda g: g.sample(10, random_state=config.SEED))
with open(config.OUTPUT / "ocr_muestra_50.txt", "w", encoding="utf-8") as f:
    for r in lectura.itertuples():
        f.write(f"\n{'=' * 90}\nID {r.id} · {r.job}\n{'=' * 90}\n{r.texto_ocr}\n")
print(open(config.OUTPUT / "ocr_muestra_50.txt", encoding="utf-8").read()[:4000])

## 6 · Descargar los entregables livianos

Baja el zip y descomprímelo **en la raíz de tu repo local** (sobrescribe). Contiene solo tablas, IDs,
la figura y los textos OCR; **ni imágenes ni pesos**.

In [ ]:
import shutil
paquete = Path("/content/entregables_01")
if paquete.exists():
    shutil.rmtree(paquete)
for rel in ["data/sample_ids.csv", "data/processed/ocr.parquet", "data/processed/muestra_meta.parquet",
            "data/processed/run_config.json", "output"]:
    origen = RAIZ / rel
    destino = paquete / rel
    destino.parent.mkdir(parents=True, exist_ok=True)
    (shutil.copytree if origen.is_dir() else shutil.copy2)(origen, destino, **({"dirs_exist_ok": True} if origen.is_dir() else {}))
shutil.make_archive("/content/entregables_01", "zip", paquete)
if EN_COLAB:
    from google.colab import files
    files.download("/content/entregables_01.zip")